# Localizer training with grouped cross-validation and clean pairwise candidate scoring

This notebook trains a **localizer** that selects the visual-memory keyframe **ahead of the observed frame** using only **geometric and visual pair features** extracted from the manual visual paths.

Design choices:
- no navigator outputs
- no heuristic admissibility rules
- no hand-crafted localization score
- grouped evaluation with **Leave-One-Log-Out**
- pairwise formulation: for each observed frame, score every keyframe candidate and select the highest-scoring one


In [1]:

import os
import json
import yaml
import joblib
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
import quaternion

from modules.rgbd_similarity import RGBDSimilarity
from modules.feature_based_point_cloud_registration import FeatureBasedPointCloudRegistration

try:
    from visual_memory_selector import VisualMemorySelector
except Exception as e:
    raise ImportError(
        "Could not import VisualMemorySelector from visual_memory_selector.py. "
        "Make sure that file is in the working directory."
    )

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


/home/rodriguez/miniconda3/envs/habitat/lib/python3.9/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/examples/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


In [ ]:

# ------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------

BASE_DIR = "/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/simple_logs"

TRAIN_LOGS = [
    {
        "name": "rep_dinning",
        "rgb": os.path.join(BASE_DIR, "rep_rep_dinning-0_224_20260328_084403_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep_dinning-0_224_20260328_084403_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep_dinning-0_224_20260328_084403_metadata.json"),
    },
    {
        "name": "rep_wc",
        "rgb": os.path.join(BASE_DIR, "rep_rep_wc-0_224_20260329_234614_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep_wc-0_224_20260329_234614_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep_wc-0_224_20260329_234614_metadata.json"),
    },
    {
        "name": "rep_kitchen-short",
        "rgb": os.path.join(BASE_DIR, "rep_rep_kitchen-0_224_20260330_110002_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep_kitchen-0_224_20260330_110002_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep_kitchen-0_224_20260330_110002_metadata.json"),
    },
    {
        "name": "rep_kitchen-long",
        "rgb": os.path.join(BASE_DIR, "rep_rep_kitchen-0_224_20260330_110101_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep_kitchen-0_224_20260330_110101_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep_kitchen-0_224_20260330_110101_metadata.json"),
    },
    {
        "name": "rep_waiting_room",
        "rgb": os.path.join(BASE_DIR, "rep_rep-1_224_20260330_122612_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep-1_224_20260330_122612_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep-1_224_20260330_122612_metadata.json"),
    },
    {
        "name": "rep_bed-to-lavatory",
        "rgb": os.path.join(BASE_DIR, "rep_rep-2_224_20260330_124559_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep-2_224_20260330_124559_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep-2_224_20260330_124559_metadata.json"),
    },
    {
        "name": "rep_left_bed", 
        "rgb": os.path.join(BASE_DIR, "rep_rep-2_left_bed_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep-2_left_bed_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep-2_left_bed_metadata.json"),
    },
    {
        "name": "rep_right_kitchen", 
        "rgb": os.path.join(BASE_DIR, "rep_rep-4_right_kitchen_rgb.npy"),
        "depth": os.path.join(BASE_DIR, "rep_rep-4_right_kitchen_depth.npy"),
        "meta": os.path.join(BASE_DIR, "rep_rep-4_right_kitchen_metadata.json"),
    },      
]

TEST_LOG = {
    "name": "rep_bed_tv",
    "rgb": os.path.join(BASE_DIR, "rep_rep_bed_tv-1_224_20260329_234304_rgb.npy"),
    "depth": os.path.join(BASE_DIR, "rep_rep_bed_tv-1_224_20260329_234304_depth.npy"),
    "meta": os.path.join(BASE_DIR, "rep_rep_bed_tv-1_224_20260329_234304_metadata.json"),
}

ALL_LOGS = TRAIN_LOGS + [TEST_LOG]

# Visual-memory selection
SIGMA_K = 1.0
SAMPLE_RATE = 1

# Pairwise-localizer dataset
INCLUDE_SELECTED_FRAME_AS_VALID_OBSERVATION = True   # selected frame still targets the next keyframe
MAX_NEGATIVES_PER_OBSERVATION = None                 # None -> keep all candidate negatives

# Feature extraction / registration config
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FEATURE_NAV_CONF = "LightGlue"
FEATURE_MODE = "mnn"
ID_RUN_FOR_FEATURES = 1201
EXAMPLES_CONFIG_PATH = "/home/rodriguez/Documents/GitHub/habitat/habitat-lab/examples/config.yaml"

# Output
OUTPUT_DIR = "/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training
RANDOM_STATE = 42
CV_STRATEGY = "LeaveOneGroupOut"

print("Device:", DEVICE)
print("Output dir:", OUTPUT_DIR)
print("CV strategy:", CV_STRATEGY)
print("Number of logs:", len(ALL_LOGS))


Device: cuda
Output dir: /home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv
CV strategy: LeaveOneGroupOut
Number of logs: 7


In [3]:

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------

@dataclass
class LoadedLog:
    name: str
    rgb_frames: np.ndarray
    depth_frames: np.ndarray
    metadata: dict
    steps: List[dict]
    selected_indices: List[int]
    selection_result: dict


def load_single_log(log_cfg: Dict, selector: VisualMemorySelector) -> LoadedLog:
    rgb_frames = np.load(log_cfg["rgb"])
    depth_frames = np.load(log_cfg["depth"])

    with open(log_cfg["meta"], "r", encoding="utf-8") as f:
        metadata = json.load(f)

    steps = metadata["steps"]
    if len(rgb_frames) != len(steps):
        raise ValueError(
            f"Mismatch in {log_cfg['name']}: "
            f"{len(rgb_frames)} rgb frames vs {len(steps)} metadata steps"
        )

    selection_result = selector.select_from_arrays(
        rgb_frames=rgb_frames,
        depth_frames=depth_frames,
        sample_rate=SAMPLE_RATE,
    )
    selected_indices = selection_result["selected_real_indices"]

    return LoadedLog(
        name=log_cfg["name"],
        rgb_frames=rgb_frames,
        depth_frames=depth_frames,
        metadata=metadata,
        steps=steps,
        selected_indices=selected_indices,
        selection_result=selection_result,
    )


def build_active_target_lookup(
    selected_indices: List[int],
    num_frames: int,
    include_selected_frame_as_valid_observation: bool = True,
) -> List[Optional[int]]:
    """
    For each raw frame, return the next visual-memory keyframe that lies ahead.

    Example with selected indices [3, 7, 10]:
    frames 0..2 -> None
    frame 3    -> 7   (if include_selected_frame_as_valid_observation=True)
    frames 4..6 -> 7
    frame 7    -> 10
    frames 8..9 -> 10
    frame 10   -> None
    """
    selected_sorted = sorted(selected_indices)
    target_lookup: List[Optional[int]] = [None] * num_frames

    if len(selected_sorted) < 2:
        return target_lookup

    for k in range(len(selected_sorted) - 1):
        start_kf = selected_sorted[k]
        next_kf = selected_sorted[k + 1]

        if include_selected_frame_as_valid_observation:
            start_frame = start_kf
        else:
            start_frame = start_kf + 1

        for i in range(start_frame, next_kf):
            if 0 <= i < num_frames:
                target_lookup[i] = next_kf

    return target_lookup


def safe_softmax_binary_decision_scores(proba: np.ndarray) -> np.ndarray:
    if proba.ndim != 2 or proba.shape[1] != 2:
        raise ValueError(f"Expected proba with shape [N, 2], got {proba.shape}")
    return proba[:, 1].astype(float)


def load_feature_registration_config() -> dict:
    with open(EXAMPLES_CONFIG_PATH, "r") as f:
        return yaml.safe_load(f)


def build_feature_extractors(device: str):
    config = load_feature_registration_config()

    rgbd_similarity = RGBDSimilarity(device=device)

    feature_registration = FeatureBasedPointCloudRegistration(
        config=config,
        device=device,
        id_run=ID_RUN_FOR_FEATURES,
        feature_nav_conf=FEATURE_NAV_CONF,
        feature_mode=FEATURE_MODE,
        topological_map=None,
        manual_operation=False,
    )

    return rgbd_similarity, feature_registration


def quaternion_to_wxyz(q_obj) -> Tuple[float, float, float, float]:
    if hasattr(q_obj, "w") and hasattr(q_obj, "x") and hasattr(q_obj, "y") and hasattr(q_obj, "z"):
        return float(q_obj.w), float(q_obj.x), float(q_obj.y), float(q_obj.z)
    arr = quaternion.as_float_array(q_obj)
    return float(arr[0]), float(arr[1]), float(arr[2]), float(arr[3])


def extract_pair_features(
    observed_rgb,
    observed_depth,
    key_rgb,
    key_depth,
    rgbd_similarity,
    feature_registration,
) -> Dict[str, Optional[float]]:
    sim_score = float(
        rgbd_similarity.compute_image_similarity(
            observed_rgb, observed_depth, key_rgb, key_depth
        )
    )

    bot_lost, est_quaternion, rmse, est_t_source_target, _ = (
        feature_registration.compute_relative_pose(
            observed_rgb, observed_depth, key_rgb, key_depth
        )
    )

    if bot_lost or est_quaternion is None or rmse is None or est_t_source_target is None:
        return {
            "ok": False,
            "error": "relative_pose_estimation_failed",
            "sim_score": sim_score,
            "rmse": np.nan,
            "tx": np.nan,
            "ty": np.nan,
            "tz": np.nan,
            "qw": np.nan,
            "qx": np.nan,
            "qy": np.nan,
            "qz": np.nan,
        }

    tx, ty, tz = [float(v) for v in np.asarray(est_t_source_target).reshape(-1)[:3]]
    qw, qx, qy, qz = quaternion_to_wxyz(est_quaternion)

    return {
        "ok": True,
        "error": None,
        "sim_score": float(sim_score),
        "rmse": float(rmse),
        "tx": float(tx),
        "ty": float(ty),
        "tz": float(tz),
        "qw": float(qw),
        "qx": float(qx),
        "qy": float(qy),
        "qz": float(qz),
    }


In [4]:

# ------------------------------------------------------------------
# LOAD LOGS AND GENERATE VISUAL MEMORIES
# ------------------------------------------------------------------

selector = VisualMemorySelector(sigma_k=SIGMA_K)

all_loaded_logs: Dict[str, LoadedLog] = {}
for log_cfg in ALL_LOGS:
    loaded = load_single_log(log_cfg, selector)
    all_loaded_logs[loaded.name] = loaded
    print(f"Loaded log: {loaded.name}")
    print(f"  frames          : {len(loaded.rgb_frames)}")
    print(f"  selected indices: {loaded.selected_indices}")
    print(f"  threshold       : {loaded.selection_result['threshold']:.4f}")


Computing similarity matrix: 100%|██████████| 266/266 [00:11<00:00, 24.01it/s]


Loaded log: rep_dinning
  frames          : 266
  selected indices: [0, 2, 4, 6, 9, 13, 17, 20, 23, 27, 29, 31, 33, 34, 36, 39, 41, 42, 44, 46, 47, 49, 51, 52, 53, 55, 57, 58, 59, 61, 62, 65, 67, 69, 71, 73, 75, 78, 80, 82, 86, 90, 95, 98, 100, 102, 105, 108, 112, 114, 116, 118, 120, 121, 122, 123, 124, 126, 127, 129, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 151, 152, 154, 156, 158, 161, 164, 166, 169, 172, 175, 177, 179, 181, 183, 187, 189, 191, 193, 195, 197, 199, 201, 203, 205, 207, 209, 211, 213, 216, 218, 220, 222, 224, 226, 228, 230, 231, 233, 234, 236, 238, 240, 242, 245, 246, 247, 248, 249, 252, 254, 256, 257, 258, 259, 260, 262, 265]
  threshold       : 0.9399


Computing similarity matrix: 100%|██████████| 298/298 [00:13<00:00, 21.32it/s] 


Loaded log: rep_wc
  frames          : 298
  selected indices: [0, 3, 6, 8, 9, 11, 13, 15, 16, 17, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 32, 34, 38, 41, 43, 45, 47, 49, 51, 53, 55, 57, 58, 61, 63, 65, 68, 70, 72, 74, 76, 78, 80, 82, 84, 85, 86, 87, 89, 91, 93, 95, 98, 100, 102, 104, 107, 109, 111, 113, 115, 117, 119, 121, 123, 125, 127, 129, 132, 135, 137, 139, 141, 143, 146, 150, 154, 156, 158, 161, 162, 166, 168, 170, 172, 175, 177, 179, 182, 184, 186, 188, 190, 192, 194, 196, 198, 200, 202, 204, 206, 208, 210, 212, 214, 217, 220, 225, 231, 235, 239, 242, 244, 247, 250, 253, 255, 257, 259, 263, 266, 269, 272, 275, 278, 280, 281, 283, 285, 287, 290, 292, 295, 297]
  threshold       : 0.9346


Computing similarity matrix: 100%|██████████| 151/151 [00:03<00:00, 41.87it/s]


...

Loaded log: rep_waiting_room
  frames          : 150
  selected indices: [0, 3, 5, 8, 11, 13, 15, 17, 19, 22, 25, 26, 27, 29, 30, 31, 33, 36, 38, 40, 43, 44, 47, 49, 51, 53, 55, 56, 58, 60, 62, 64, 66, 68, 69, 71, 72, 74, 75, 76, 78, 82, 85, 87, 89, 93, 95, 97, 99, 101, 103, 105, 107, 110, 113, 115, 117, 119, 121, 125, 127, 130, 131, 133, 135, 137, 138, 140, 142, 143, 144, 146, 148, 149]
  threshold       : 0.9406


Computing similarity matrix: 100%|██████████| 116/116 [00:02<00:00, 53.88it/s]


Loaded log: rep_bed-to-lavatory
  frames          : 116
  selected indices: [0, 3, 4, 6, 8, 10, 13, 14, 15, 17, 21, 25, 29, 32, 35, 38, 41, 43, 45, 46, 47, 48, 50, 52, 54, 56, 59, 60, 63, 66, 69, 72, 73, 74, 76, 77, 78, 80, 83, 85, 87, 89, 92, 95, 98, 101, 104, 107, 110, 113, 115]
  threshold       : 0.9288


Computing similarity matrix: 100%|██████████| 225/225 [00:07<00:00, 28.36it/s] 

Loaded log: rep_bed_tv
  frames          : 225
  selected indices: [0, 2, 4, 6, 7, 9, 11, 13, 16, 18, 19, 21, 23, 26, 28, 30, 32, 34, 35, 37, 40, 43, 46, 48, 50, 52, 54, 56, 59, 61, 63, 64, 66, 68, 69, 71, 72, 73, 74, 75, 79, 81, 82, 84, 86, 87, 88, 89, 90, 93, 97, 101, 103, 105, 107, 111, 114, 116, 118, 120, 122, 124, 125, 127, 129, 130, 132, 134, 136, 138, 139, 141, 143, 145, 146, 147, 149, 151, 153, 155, 157, 160, 163, 166, 170, 173, 177, 180, 183, 187, 191, 195, 198, 202, 205, 207, 210, 212, 214, 217, 219, 223, 224]
  threshold       : 0.9384


In [5]:

# ------------------------------------------------------------------
# VISUAL-MEMORY SUMMARY TABLE
# ------------------------------------------------------------------

summary_rows = []
for name, log_obj in all_loaded_logs.items():
    summary_rows.append({
        "log": name,
        "num_frames": len(log_obj.rgb_frames),
        "num_keyframes": len(log_obj.selected_indices),
        "first_keyframe": None if len(log_obj.selected_indices) == 0 else int(log_obj.selected_indices[0]),
        "last_keyframe": None if len(log_obj.selected_indices) == 0 else int(log_obj.selected_indices[-1]),
        "threshold": log_obj.selection_result["threshold"],
        "mu_d1": log_obj.selection_result["mu"],
        "sigma_d1": log_obj.selection_result["sigma"],
    })

selection_summary_df = pd.DataFrame(summary_rows).sort_values("log").reset_index(drop=True)
display(selection_summary_df)


,log,num_frames,num_keyframes,first_keyframe,last_keyframe,threshold,mu_d1,sigma_d1
0,rep_bed-to-lavatory,116,51,0,115,0.928842,0.945744,0.016903
1,rep_bed_tv,225,103,0,224,0.938441,0.952792,0.014351
2,rep_dinning,266,128,0,265,0.939857,0.951678,0.011821
3,rep_kitchen-long,254,124,0,253,0.947405,0.958646,0.011240
4,rep_kitchen-short,151,73,0,150,0.946981,0.957751,0.010770
5,rep_waiting_room,150,74,0,149,0.940616,0.951041,0.010425
6,rep_wc,298,135,0,297,0.934632,0.949984,0.015353


In [6]:

# ------------------------------------------------------------------
# BUILD OBSERVATION -> TRUE TARGET TABLE
# ------------------------------------------------------------------

target_rows = []
for name, log_obj in all_loaded_logs.items():
    num_frames = len(log_obj.rgb_frames)
    target_lookup = build_active_target_lookup(
        selected_indices=log_obj.selected_indices,
        num_frames=num_frames,
        include_selected_frame_as_valid_observation=INCLUDE_SELECTED_FRAME_AS_VALID_OBSERVATION,
    )

    selected_sorted = sorted(log_obj.selected_indices)
    selected_rank_map = {kf: rank for rank, kf in enumerate(selected_sorted)}

    for frame_id, target_kf in enumerate(target_lookup):
        if target_kf is None:
            continue

        target_rows.append({
            "source_log": name,
            "raw_frame_id": int(frame_id),
            "target_keyframe": int(target_kf),
            "target_rank": int(selected_rank_map[target_kf]),
            "num_keyframes_in_log": int(len(selected_sorted)),
        })

target_df = pd.DataFrame(target_rows)
print("Observation-target rows:", target_df.shape)
display(target_df.head(20))


Observation-target rows: (1453, 5)


,source_log,raw_frame_id,target_keyframe,target_rank,num_keyframes_in_log
0,rep_dinning,0,2,1,128
1,rep_dinning,1,2,1,128
2,rep_dinning,2,4,2,128
3,rep_dinning,3,4,2,128
4,rep_dinning,4,6,3,128
5,rep_dinning,5,6,3,128
6,rep_dinning,6,9,4,128
7,rep_dinning,7,9,4,128
8,rep_dinning,8,9,4,128
9,rep_dinning,9,13,5,128


In [7]:

# ------------------------------------------------------------------
# TARGET DISTRIBUTION SUMMARY
# ------------------------------------------------------------------

print("Rows per source log:")
display(target_df.groupby("source_log").size().to_frame("num_observations"))

print("Target rank histogram per log:")
display(
    target_df.groupby(["source_log", "target_rank"]).size().unstack(fill_value=0)
)


Rows per source log:


,num_observations
source_log,
rep_bed-to-lavatory,115
rep_bed_tv,224
rep_dinning,265
rep_kitchen-long,253
rep_kitchen-short,150
rep_waiting_room,149
rep_wc,297


Target rank histogram per log:


target_rank,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134
source_log,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
rep_bed-to-lavatory,3,1,2,2,2,3,1,1,2,4,4,4,3,3,3,3,2,2,1,1,1,2,2,2,2,3,1,3,3,3,3,1,1,2,1,1,2,3,2,2,2,3,3,3,3,3,3,3,3,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
rep_bed_tv,2,2,2,1,2,2,2,3,2,1,2,2,3,2,2,2,2,1,2,3,3,3,2,2,2,2,2,3,2,2,1,2,2,1,2,1,1,1,1,4,2,1,2,2,1,1,1,1,3,4,4,2,2,2,4,3,2,2,2,2,2,1,2,2,1,2,2,2,2,1,2,2,2,1,1,2,2,2,2,2,3,3,3,4,3,4,3,3,4,4,4,3,4,3,2,3,2,2,3,2,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
rep_dinning,2,2,2,3,4,4,3,3,4,2,2,2,1,2,3,2,1,2,2,1,2,2,1,1,2,2,1,1,2,1,3,2,2,2,2,2,3,2,2,4,4,5,3,2,2,3,3,4,2,2,2,2,1,1,1,1,2,1,2,3,2,2,2,2,2,2,2,2,2,1,1,2,2,2,3,3,2,3,3,3,2,2,2,2,4,2,2,2,2,2,2,2,2,2,2,2,2,2,3,2,2,2,2,2,2,2,1,2,1,2,2,2,2,3,1,1,1,1,3,2,2,1,1,1,1,2,3,0,0,0,0,0,0,0
rep_kitchen-long,3,2,1,1,2,2,1,1,2,2,1,1,1,1,2,2,4,3,3,3,3,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,2,2,2,2,3,2,2,2,2,3,2,3,3,3,2,3,2,2,2,2,1,1,1,2,2,2,1,1,1,1,1,2,1,2,3,2,1,1,2,1,1,2,2,2,3,2,1,2,2,1,2,2,2,2,2,2,2,2,2,2,1,3,3,2,3,3,4,2,4,3,4,3,3,3,2,2,2,2,2,2,1,1,3,2,2,0,0,0,0,0,0,0,0,0,0,0
rep_kitchen-short,2,3,2,2,1,2,1,2,1,1,1,1,2,1,1,3,1,2,4,2,3,2,3,4,4,4,3,3,2,2,3,2,2,2,2,3,2,1,2,2,2,2,3,2,2,2,2,2,2,2,3,2,2,2,2,3,2,1,2,2,2,2,2,2,3,2,2,2,2,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
rep_waiting_room,3,2,3,3,2,2,2,2,3,3,1,1,2,1,1,2,3,2,2,3,1,3,2,2,2,2,1,2,2,2,2,2,2,1,2,1,2,1,1,2,4,3,2,2,4,2,2,2,2,2,2,2,3,3,2,2,2,2,4,2,3,1,2,2,2,1,2,2,1,1,2,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
rep_wc,3,3,2,1,2,2,2,1,1,2,2,1,1,1,1,1,1,1,1,1,2,2,4,3,2,2,2,2,2,2,2,2,1,3,2,2,3,2,2,2,2,2,2,2,2,1,1,1,2,2,2,2,3,2,2,2,3,2,2,2,2,2,2,2,2,2,2,2,3,3,2,2,2,2,3,4,4,2,2,3,1,4,2,2,2,3,2,2,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,3,3,5,6,4,4,3,2,3,3,3,2,2,2,4,3,3,3,3,3,2,1,2,2,2,3,2,3,2


In [8]:

# ------------------------------------------------------------------
# BUILD FEATURE EXTRACTORS
# ------------------------------------------------------------------

rgbd_similarity, feature_registration = build_feature_extractors(DEVICE)
print("Feature extractors ready.")


Feature extractors ready.


In [9]:

# ------------------------------------------------------------------
# EXTRACT PAIRWISE CANDIDATE FEATURES FOR ALL OBSERVATIONS AND ALL KEYFRAMES
# ------------------------------------------------------------------

pair_rows = []

for log_name, log_obj in all_loaded_logs.items():
    selected_sorted = sorted(log_obj.selected_indices)
    target_lookup = build_active_target_lookup(
        selected_indices=selected_sorted,
        num_frames=len(log_obj.rgb_frames),
        include_selected_frame_as_valid_observation=INCLUDE_SELECTED_FRAME_AS_VALID_OBSERVATION,
    )
    selected_rank_map = {kf: rank for rank, kf in enumerate(selected_sorted)}

    valid_frames = [i for i, tgt in enumerate(target_lookup) if tgt is not None]
    iterator = tqdm(valid_frames, desc=f"Extracting pairs | {log_name}")

    for raw_frame_id in iterator:
        target_kf = int(target_lookup[raw_frame_id])
        target_rank = int(selected_rank_map[target_kf])

        observed_rgb = log_obj.rgb_frames[raw_frame_id]
        observed_depth = log_obj.depth_frames[raw_frame_id]

        candidate_keyframes = list(selected_sorted)
        if MAX_NEGATIVES_PER_OBSERVATION is not None:
            negatives = [kf for kf in candidate_keyframes if kf != target_kf]
            rng = np.random.RandomState(RANDOM_STATE + raw_frame_id)
            negatives = list(rng.choice(
                negatives,
                size=min(MAX_NEGATIVES_PER_OBSERVATION, len(negatives)),
                replace=False
            ))
            candidate_keyframes = sorted(negatives + [target_kf])

        for candidate_kf in candidate_keyframes:
            candidate_rank = int(selected_rank_map[candidate_kf])

            feat = extract_pair_features(
                observed_rgb=observed_rgb,
                observed_depth=observed_depth,
                key_rgb=log_obj.rgb_frames[candidate_kf],
                key_depth=log_obj.depth_frames[candidate_kf],
                rgbd_similarity=rgbd_similarity,
                feature_registration=feature_registration,
            )

            pair_rows.append({
                "source_log": log_name,
                "raw_frame_id": int(raw_frame_id),
                "target_keyframe": int(target_kf),
                "target_rank": int(target_rank),
                "candidate_keyframe": int(candidate_kf),
                "candidate_rank": int(candidate_rank),
                "rank_error": int(candidate_rank - target_rank),
                "abs_rank_error": int(abs(candidate_rank - target_rank)),
                "is_positive": int(candidate_kf == target_kf),
                **feat,
            })

pair_df = pd.DataFrame(pair_rows)
print("Pairwise feature table shape:", pair_df.shape)
display(pair_df.head(20))


Extracting pairs | rep_dinning: 100%|██████████| 265/265 [1:39:14<00:00, 22.47s/it]
Extracting pairs | rep_wc: 100%|██████████| 297/297 [2:01:16<00:00, 24.50s/it]  
Extracting pairs | rep_kitchen-short: 100%|██████████| 150/150 [31:41<00:00, 12.68s/it]
Extracting pairs | rep_kitchen-long: 100%|██████████| 253/253 [1:31:37<00:00, 21.73s/it]
Extracting pairs | rep_waiting_room: 100%|██████████| 149/149 [32:22<00:00, 13.04s/it]
Extracting pairs | rep_bed-to-lavatory: 100%|██████████| 115/115 [17:08<00:00,  8.95s/it]
Extracting pairs | rep_bed_tv: 100%|██████████| 224/224 [1:06:47<00:00, 17.89s/it]


Pairwise feature table shape: (156300, 20)


,source_log,raw_frame_id,target_keyframe,target_rank,candidate_keyframe,candidate_rank,rank_error,abs_rank_error,is_positive,ok,error,sim_score,rmse,tx,ty,tz,qw,qx,qy,qz
0,rep_dinning,0,2,1,0,0,-1,1,0,True,None,1.000000,7.971944e-16,-1.110223e-16,7.771561e-16,-8.881784e-16,-1.000000,7.793685e-17,-2.010076e-16,-9.748082e-17
1,rep_dinning,0,2,1,2,1,0,0,1,True,None,0.932823,2.253966e-01,-1.104259e-02,2.779756e-02,1.710790e-01,-0.999988,3.831845e-03,2.790258e-03,1.237332e-03
2,rep_dinning,0,2,1,4,2,1,1,0,True,None,0.901132,2.881738e-01,-2.046455e-03,-3.504616e-02,3.467045e-01,-0.999991,-2.705519e-03,3.294855e-03,5.246998e-04
3,rep_dinning,0,2,1,6,3,2,2,0,True,None,0.879928,2.957069e-01,-5.480281e-02,1.123640e-03,5.437215e-01,-0.999969,1.280867e-03,7.779254e-03,-8.775986e-04
4,rep_dinning,0,2,1,9,4,3,3,0,True,None,0.856264,5.354307e-01,-2.313571e-01,6.426863e-02,7.779583e-01,-0.999548,6.924698e-03,2.867771e-02,5.786128e-03
5,rep_dinning,0,2,1,13,5,4,4,0,True,None,0.829208,5.975867e-01,-2.315248e-01,-5.803951e-02,9.420899e-01,-0.999985,-4.960048e-03,2.421759e-03,-1.360546e-04
6,rep_dinning,0,2,1,17,6,5,5,0,True,None,0.818510,5.889780e-01,-1.468604e-01,-3.880623e-02,1.231687e+00,0.999298,3.380348e-03,3.730580e-02,-1.032129e-03
7,rep_dinning,0,2,1,20,7,6,6,0,True,None,0.808824,5.814960e-01,-1.331513e-01,-1.708395e-01,1.202277e+00,0.997893,1.901468e-02,6.199799e-02,-2.097621e-03
8,rep_dinning,0,2,1,23,8,7,7,0,True,None,0.795363,4.463893e-01,9.272664e-02,1.148022e-01,1.478521e+00,0.995105,-1.325116e-02,9.769789e-02,-6.685768e-03
9,rep_dinning,0,2,1,27,9,8,8,0,True,None,0.800558,6.748899e-01,-5.638288e-02,-1.364433e-01,1.486850e+00,0.992782,1.305007e-02,1.192126e-01,1.339773e-03


In [10]:

# ------------------------------------------------------------------
# CLEAN FEATURE TABLE AND SAVE IT
# ------------------------------------------------------------------

feature_cols = ["rmse", "tx", "ty", "tz", "qw", "qx", "qy", "qz", "sim_score"]

pair_df_clean = pair_df.dropna(subset=feature_cols).reset_index(drop=True)
pair_df_clean["is_positive"] = pair_df_clean["is_positive"].astype(int)

pair_csv = os.path.join(OUTPUT_DIR, "all_localizer_pair_features.csv")
pair_df_clean.to_csv(pair_csv, index=False)

print("Clean pairwise feature shape:", pair_df_clean.shape)
print("Saved:", pair_csv)


Clean pairwise feature shape: (156263, 20)
Saved: /home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/all_localizer_pair_features.csv


In [11]:

# ------------------------------------------------------------------
# FEATURE-TABLE SUMMARY
# ------------------------------------------------------------------

print("Class counts:")
display(pair_df_clean["is_positive"].value_counts().rename(index={0: "negative", 1: "positive"}).to_frame("count"))

print("Rows by source log:")
display(pair_df_clean.groupby("source_log").size().to_frame("num_pair_rows"))

print("Mean number of candidates per observation:")
obs_counts_df = (
    pair_df_clean.groupby(["source_log", "raw_frame_id"]).size().reset_index(name="num_candidates")
)
display(obs_counts_df.groupby("source_log")["num_candidates"].mean().to_frame("mean_num_candidates"))


Class counts:


,count
is_positive,
negative,154810
positive,1453


Rows by source log:


,num_pair_rows
source_log,
rep_bed-to-lavatory,5859
rep_bed_tv,23071
rep_dinning,33908
rep_kitchen-long,31357
rep_kitchen-short,10950
rep_waiting_room,11026
rep_wc,40092


Mean number of candidates per observation:


,mean_num_candidates
source_log,
rep_bed-to-lavatory,50.947826
rep_bed_tv,102.995536
rep_dinning,127.954717
rep_kitchen-long,123.940711
rep_kitchen-short,73.000000
rep_waiting_room,74.000000
rep_wc,134.989899


In [12]:

# ------------------------------------------------------------------
# PREPARE GROUPED CROSS-VALIDATION
# ------------------------------------------------------------------

X_all = pair_df_clean[feature_cols].values.astype(np.float32)
y_all = pair_df_clean["is_positive"].values.astype(int)
groups = pair_df_clean["source_log"].values

logo = LeaveOneGroupOut()
fold_plan = []
for fold_id, (_, val_idx) in enumerate(logo.split(X_all, y_all, groups), start=1):
    heldout_logs = sorted(set(groups[val_idx]))
    fold_plan.append({
        "fold": fold_id,
        "heldout_log": heldout_logs[0],
        "num_val_rows": int(len(val_idx)),
        "num_val_observations": int(pair_df_clean.iloc[val_idx][["source_log", "raw_frame_id"]].drop_duplicates().shape[0]),
    })

fold_plan_df = pd.DataFrame(fold_plan)
print("Total pair rows     :", len(X_all))
print("Total folds         :", len(fold_plan_df))
print("Positive ratio      :", round(float(y_all.mean()), 6))
display(fold_plan_df)


Total pair rows     : 156263
Total folds         : 7
Positive ratio      : 0.009298


,fold,heldout_log,num_val_rows,num_val_observations
0,1,rep_bed-to-lavatory,5859,115
1,2,rep_bed_tv,23071,224
2,3,rep_dinning,33908,265
3,4,rep_kitchen-long,31357,253
4,5,rep_kitchen-short,10950,150
5,6,rep_waiting_room,11026,149
6,7,rep_wc,40092,297


## Model selection strategy

Each model is trained as a **binary pair scorer**:

- input: pair features between an observed frame and one candidate keyframe
- target: `1` if the candidate is the true keyframe ahead, else `0`

At validation time, the model scores **all candidates of each observation**, and the final predicted localizer index is the candidate with the highest positive probability.

This means model selection is based on **localization performance**, not just row-wise binary accuracy.


In [13]:

# ------------------------------------------------------------------
# TRAINING / EVALUATION HELPERS
# ------------------------------------------------------------------

def oversample_positive_rows(X: np.ndarray, y: np.ndarray, random_state: int, target_pos_ratio: float = 0.33):
    """
    Simple training-only oversampling for models without class_weight support.
    """
    rng = np.random.RandomState(random_state)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return X, y

    current_ratio = len(pos_idx) / len(y)
    if current_ratio >= target_pos_ratio:
        return X, y

    desired_num_pos = int(np.ceil(target_pos_ratio * len(neg_idx) / (1.0 - target_pos_ratio)))
    num_extra = max(0, desired_num_pos - len(pos_idx))
    if num_extra == 0:
        return X, y

    sampled_pos_idx = rng.choice(pos_idx, size=num_extra, replace=True)
    X_aug = np.concatenate([X, X[sampled_pos_idx]], axis=0)
    y_aug = np.concatenate([y, y[sampled_pos_idx]], axis=0)
    return X_aug, y_aug


def make_candidate_model_zoo(random_state: int = RANDOM_STATE):
    return [
        {
            "name": "logreg_balanced",
            "estimator": LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=1500,
                solver="lbfgs",
                random_state=random_state,
            ),
            "use_scaler": True,
            "use_positive_oversampling": False,
        },
        {
            "name": "rf_balanced",
            "estimator": RandomForestClassifier(
                n_estimators=400,
                max_depth=None,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=random_state,
            ),
            "use_scaler": False,
            "use_positive_oversampling": False,
        },
        {
            "name": "mlp_128_64_os",
            "estimator": MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                solver="adam",
                alpha=1e-4,
                learning_rate_init=5e-4,
                max_iter=600,
                random_state=random_state,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=20,
            ),
            "use_scaler": True,
            "use_positive_oversampling": True,
        },
    ]


def fit_binary_pair_model(
    train_df: pd.DataFrame,
    feature_cols: List[str],
    model_cfg: Dict,
    random_state: int = RANDOM_STATE,
):
    X_train = train_df[feature_cols].values.astype(np.float32)
    y_train = train_df["is_positive"].values.astype(int)

    scaler = None
    if model_cfg["use_positive_oversampling"]:
        X_train, y_train = oversample_positive_rows(
            X_train, y_train, random_state=random_state, target_pos_ratio=0.33
        )

    if model_cfg["use_scaler"]:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)

    estimator = clone(model_cfg["estimator"])
    estimator.fit(X_train, y_train)
    return estimator, scaler


def predict_pair_scores(
    model,
    scaler,
    df: pd.DataFrame,
    feature_cols: List[str],
) -> np.ndarray:
    X = df[feature_cols].values.astype(np.float32)
    if scaler is not None:
        X = scaler.transform(X)

    proba = model.predict_proba(X)
    return safe_softmax_binary_decision_scores(proba)


def evaluate_localizer_on_pair_df(
    scored_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """
    scored_df must contain:
      - source_log
      - raw_frame_id
      - target_keyframe
      - target_rank
      - candidate_keyframe
      - candidate_rank
      - is_positive
      - score
    """
    pred_rows = []
    grouped = scored_df.groupby(["source_log", "raw_frame_id"], sort=False)

    for (source_log, raw_frame_id), g in grouped:
        g = g.sort_values(["score", "candidate_rank"], ascending=[False, True]).reset_index(drop=True)
        best = g.iloc[0]

        target_rank = int(best["target_rank"])
        pred_rank = int(best["candidate_rank"])
        abs_rank_error = abs(pred_rank - target_rank)

        positive_row = g[g["is_positive"] == 1]
        if len(positive_row) != 1:
            raise ValueError(
                f"Expected exactly one positive row for {(source_log, raw_frame_id)}, got {len(positive_row)}"
            )
        positive_score = float(positive_row.iloc[0]["score"])
        rank_position = int((g["score"] > positive_score).sum()) + 1

        pred_rows.append({
            "source_log": source_log,
            "raw_frame_id": int(raw_frame_id),
            "target_keyframe": int(best["target_keyframe"]),
            "target_rank": int(target_rank),
            "pred_candidate_keyframe": int(best["candidate_keyframe"]),
            "pred_candidate_rank": int(pred_rank),
            "pred_score": float(best["score"]),
            "is_exact_match": bool(int(best["candidate_keyframe"]) == int(best["target_keyframe"])),
            "is_within_1_rank": bool(abs_rank_error <= 1),
            "abs_rank_error": int(abs_rank_error),
            "mrr": float(1.0 / rank_position),
        })

    pred_df = pd.DataFrame(pred_rows).sort_values(["source_log", "raw_frame_id"]).reset_index(drop=True)

    metrics = {
        "num_observations": int(len(pred_df)),
        "top1_exact_accuracy": float(pred_df["is_exact_match"].mean()),
        "top1_within1_accuracy": float(pred_df["is_within_1_rank"].mean()),
        "mean_abs_rank_error": float(pred_df["abs_rank_error"].mean()),
        "median_abs_rank_error": float(pred_df["abs_rank_error"].median()),
        "mrr": float(pred_df["mrr"].mean()),
    }
    return pred_df, metrics


In [14]:

# ------------------------------------------------------------------
# TRAIN MODEL ZOO WITH LEAVE-ONE-LOG-OUT CROSS-VALIDATION
# ------------------------------------------------------------------

candidate_models = make_candidate_model_zoo(random_state=RANDOM_STATE)

model_selection_rows = []
cv_artifacts = {}

for model_cfg in candidate_models:
    print("=" * 90)
    print(f"Evaluating localizer model: {model_cfg['name']}")
    print("=" * 90)

    fold_rows = []
    fold_prediction_tables = []

    # Optional row-level OOF scores
    oof_scores = np.full(len(pair_df_clean), np.nan, dtype=np.float32)

    for fold_id, (train_idx, val_idx) in enumerate(logo.split(X_all, y_all, groups), start=1):
        train_df = pair_df_clean.iloc[train_idx].reset_index(drop=True)
        val_df = pair_df_clean.iloc[val_idx].reset_index(drop=True)
        heldout_log = str(val_df["source_log"].iloc[0])

        model, scaler = fit_binary_pair_model(
            train_df=train_df,
            feature_cols=feature_cols,
            model_cfg=model_cfg,
            random_state=RANDOM_STATE + fold_id,
        )

        val_scores = predict_pair_scores(
            model=model,
            scaler=scaler,
            df=val_df,
            feature_cols=feature_cols,
        )
        oof_scores[val_idx] = val_scores

        val_scored_df = val_df.copy()
        val_scored_df["score"] = val_scores

        pred_df, localizer_metrics = evaluate_localizer_on_pair_df(val_scored_df)

        fold_row = {
            "fold": int(fold_id),
            "heldout_log": heldout_log,
            **localizer_metrics,
        }

        # Candidate-level binary metrics are secondary, but useful for diagnostics.
        try:
            fold_row["pair_auc_roc"] = float(roc_auc_score(val_scored_df["is_positive"].values, val_scores))
        except Exception:
            fold_row["pair_auc_roc"] = np.nan
        try:
            fold_row["pair_average_precision"] = float(average_precision_score(val_scored_df["is_positive"].values, val_scores))
        except Exception:
            fold_row["pair_average_precision"] = np.nan

        fold_rows.append(fold_row)

        pred_df["fold"] = int(fold_id)
        pred_df["heldout_log"] = heldout_log
        fold_prediction_tables.append(pred_df)

        print(
            f"Fold {fold_id:02d} | heldout={heldout_log:20s} | "
            f"top1_exact={localizer_metrics['top1_exact_accuracy']:.4f} | "
            f"within1={localizer_metrics['top1_within1_accuracy']:.4f} | "
            f"mae_rank={localizer_metrics['mean_abs_rank_error']:.4f} | "
            f"mrr={localizer_metrics['mrr']:.4f}"
        )

    fold_results_df = pd.DataFrame(fold_rows)
    oof_observation_df = pd.concat(fold_prediction_tables, ignore_index=True)

    summary_row = {
        "name": model_cfg["name"],
        "mean_top1_exact_accuracy": float(fold_results_df["top1_exact_accuracy"].mean()),
        "mean_top1_within1_accuracy": float(fold_results_df["top1_within1_accuracy"].mean()),
        "mean_mae_rank": float(fold_results_df["mean_abs_rank_error"].mean()),
        "mean_mrr": float(fold_results_df["mrr"].mean()),
        "mean_pair_auc_roc": float(fold_results_df["pair_auc_roc"].mean()),
        "mean_pair_average_precision": float(fold_results_df["pair_average_precision"].mean()),
    }
    model_selection_rows.append(summary_row)

    cv_artifacts[model_cfg["name"]] = {
        "config": model_cfg,
        "fold_results_df": fold_results_df,
        "oof_observation_df": oof_observation_df,
        "oof_scores": oof_scores,
    }

results_df = pd.DataFrame(model_selection_rows).sort_values(
    ["mean_top1_exact_accuracy", "mean_top1_within1_accuracy", "mean_mrr", "mean_mae_rank"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(results_df)


Evaluating localizer model: logreg_balanced
Fold 01 | heldout=rep_bed-to-lavatory  | top1_exact=0.2174 | within1=1.0000 | mae_rank=0.7826 | mrr=0.5757
Fold 02 | heldout=rep_bed_tv           | top1_exact=0.2500 | within1=1.0000 | mae_rank=0.7500 | mrr=0.5923
Fold 03 | heldout=rep_dinning          | top1_exact=0.2302 | within1=1.0000 | mae_rank=0.7698 | mrr=0.5836
Fold 04 | heldout=rep_kitchen-long     | top1_exact=0.2411 | within1=1.0000 | mae_rank=0.7589 | mrr=0.5765
Fold 05 | heldout=rep_kitchen-short    | top1_exact=0.2400 | within1=1.0000 | mae_rank=0.7600 | mrr=0.5822
Fold 06 | heldout=rep_waiting_room     | top1_exact=0.2282 | within1=1.0000 | mae_rank=0.7718 | mrr=0.5727
Fold 07 | heldout=rep_wc               | top1_exact=0.2727 | within1=1.0000 | mae_rank=0.7273 | mrr=0.6010
Evaluating localizer model: rf_balanced
Fold 01 | heldout=rep_bed-to-lavatory  | top1_exact=0.6522 | within1=0.9217 | mae_rank=0.4261 | mrr=0.8232
Fold 02 | heldout=rep_bed_tv           | top1_exact=0.8750 |

,name,mean_top1_exact_accuracy,mean_top1_within1_accuracy,mean_mae_rank,mean_mrr,mean_pair_auc_roc,mean_pair_average_precision
0,rf_balanced,0.817910,0.934785,0.306071,0.905853,0.997131,0.827195
1,mlp_128_64_os,0.807584,0.936604,0.256770,0.900250,0.997730,0.815756
2,logreg_balanced,0.239943,1.000000,0.760057,0.583423,0.987757,0.291422


In [15]:

# ------------------------------------------------------------------
# SELECT BEST MODEL BASED ON GROUPED LOCALIZATION PERFORMANCE
# ------------------------------------------------------------------

best_model_name = results_df.iloc[0]["name"]
best_cfg = cv_artifacts[best_model_name]["config"]
best_cv_fold_results_df = cv_artifacts[best_model_name]["fold_results_df"]
best_oof_observation_df = cv_artifacts[best_model_name]["oof_observation_df"]
best_oof_scores = cv_artifacts[best_model_name]["oof_scores"]

print("Best localizer model:", best_model_name)
display(results_df.iloc[[0]])
display(best_cv_fold_results_df)


Best localizer model: rf_balanced


,name,mean_top1_exact_accuracy,mean_top1_within1_accuracy,mean_mae_rank,mean_mrr,mean_pair_auc_roc,mean_pair_average_precision
0,rf_balanced,0.81791,0.934785,0.306071,0.905853,0.997131,0.827195


,fold,heldout_log,num_observations,top1_exact_accuracy,top1_within1_accuracy,mean_abs_rank_error,median_abs_rank_error,mrr,pair_auc_roc,pair_average_precision
0,1,rep_bed-to-lavatory,115,0.652174,0.921739,0.426087,0.0,0.823188,0.993059,0.668025
1,2,rep_bed_tv,224,0.875000,0.941964,0.183036,0.0,0.936756,0.998823,0.891817
2,3,rep_dinning,265,0.701887,0.867925,0.430189,0.0,0.846541,0.997926,0.789409
3,4,rep_kitchen-long,253,0.841897,0.932806,0.229249,0.0,0.912846,0.998171,0.815265
4,5,rep_kitchen-short,150,0.926667,0.986667,0.086667,0.0,0.960000,0.998786,0.912265
5,6,rep_waiting_room,149,0.859060,0.959732,0.181208,0.0,0.927293,0.997595,0.844741
6,7,rep_wc,297,0.868687,0.932660,0.606061,0.0,0.934343,0.995560,0.868842


In [16]:

# ------------------------------------------------------------------
# FIT BEST MODEL ON ALL DATA AND SAVE MODEL / SCALER / METADATA
# ------------------------------------------------------------------

best_model, best_scaler = fit_binary_pair_model(
    train_df=pair_df_clean,
    feature_cols=feature_cols,
    model_cfg=best_cfg,
    random_state=RANDOM_STATE,
)

model_path = os.path.join(OUTPUT_DIR, f"localizer_{best_model_name}_cv.joblib")
scaler_path = os.path.join(OUTPUT_DIR, f"localizer_scaler_{best_model_name}_cv.joblib")
results_path = os.path.join(OUTPUT_DIR, "cv_model_selection_results.csv")
fold_results_path = os.path.join(OUTPUT_DIR, f"{best_model_name}_fold_results.csv")
oof_obs_path = os.path.join(OUTPUT_DIR, f"{best_model_name}_oof_observation_predictions.csv")
pair_oof_path = os.path.join(OUTPUT_DIR, f"{best_model_name}_pair_oof_scores.csv")
config_path = os.path.join(OUTPUT_DIR, "training_config_cv.json")

joblib.dump(best_model, model_path)
joblib.dump(best_scaler, scaler_path)
results_df.to_csv(results_path, index=False)
best_cv_fold_results_df.to_csv(fold_results_path, index=False)
best_oof_observation_df.to_csv(oof_obs_path, index=False)

pair_oof_df = pair_df_clean.copy()
pair_oof_df["oof_score"] = best_oof_scores
pair_oof_df.to_csv(pair_oof_path, index=False)

training_config = {
    "all_logs": ALL_LOGS,
    "sigma_k": SIGMA_K,
    "sample_rate": SAMPLE_RATE,
    "include_selected_frame_as_valid_observation": INCLUDE_SELECTED_FRAME_AS_VALID_OBSERVATION,
    "max_negatives_per_observation": MAX_NEGATIVES_PER_OBSERVATION,
    "feature_cols": feature_cols,
    "cv_strategy": CV_STRATEGY,
    "random_state": RANDOM_STATE,
    "feature_nav_conf": FEATURE_NAV_CONF,
    "feature_mode": FEATURE_MODE,
    "id_run_for_features": ID_RUN_FOR_FEATURES,
    "best_model_name": best_model_name,
    "best_model_config": {
        k: v for k, v in best_cfg.items() if k != "estimator"
    },
    "visual_memory_indices_by_log": {
        name: list(map(int, log_obj.selected_indices)) for name, log_obj in all_loaded_logs.items()
    },
    "notes": (
        "Clean localizer training. Pairwise binary scorer over observation-candidate pairs. "
        "No navigator outputs. No admissibility heuristics. Final localizer index is argmax score over candidates."
    ),
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2)

print("Saved:")
print(model_path)
print(scaler_path)
print(results_path)
print(fold_results_path)
print(oof_obs_path)
print(pair_oof_path)
print(config_path)


Saved:
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/localizer_rf_balanced_cv.joblib
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/localizer_scaler_rf_balanced_cv.joblib
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/cv_model_selection_results.csv
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/rf_balanced_fold_results.csv
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/rf_balanced_oof_observation_predictions.csv
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/rf_balanced_pair_oof_scores.csv
/home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/training_config_cv.json


In [17]:

# ------------------------------------------------------------------
# FINAL MODEL TRAINING CURVES (ONLY IF THE BEST MODEL IS MLP)
# ------------------------------------------------------------------

if hasattr(best_model, "loss_curve_"):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(best_model.loss_curve_, marker="o")
    plt.title(f"Loss curve - {best_model_name}")
    plt.xlabel("Iteration")
    plt.ylabel("Training loss")
    plt.grid(True)

    plt.subplot(1, 2, 2)
    if hasattr(best_model, "validation_scores_") and best_model.validation_scores_ is not None:
        plt.plot(best_model.validation_scores_, marker="o")
        plt.title(f"Internal validation score curve - {best_model_name}")
        plt.xlabel("Iteration")
        plt.ylabel("Validation score")
        plt.grid(True)
    else:
        plt.text(0.5, 0.5, "No validation scores available", ha="center", va="center")
        plt.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not expose loss_curve_.")


rf_balanced does not expose loss_curve_.


In [18]:

# ------------------------------------------------------------------
# OUT-OF-FOLD LOCALIZATION PERFORMANCE OF BEST MODEL
# ------------------------------------------------------------------

cv_metrics = {
    "top1_exact_accuracy": float(best_oof_observation_df["is_exact_match"].mean()),
    "top1_within1_accuracy": float(best_oof_observation_df["is_within_1_rank"].mean()),
    "mean_abs_rank_error": float(best_oof_observation_df["abs_rank_error"].mean()),
    "median_abs_rank_error": float(best_oof_observation_df["abs_rank_error"].median()),
    "mrr": float(best_oof_observation_df["mrr"].mean()),
}

print("Grouped OOF localization metrics")
for k, v in cv_metrics.items():
    print(f"{k:24s}: {v:.4f}")

cv_metrics_path = os.path.join(OUTPUT_DIR, "best_localizer_cv_metrics.json")
with open(cv_metrics_path, "w", encoding="utf-8") as f:
    json.dump(cv_metrics, f, indent=2)
print("Saved CV metrics to:", cv_metrics_path)

display(best_cv_fold_results_df)


Grouped OOF localization metrics
top1_exact_accuracy     : 0.8224
top1_within1_accuracy   : 0.9298
mean_abs_rank_error     : 0.3317
median_abs_rank_error   : 0.0000
mrr                     : 0.9081
Saved CV metrics to: /home/rodriguez/Documents/GitHub/habitat/habitat-lab/manual_operation/trained_localizer_outputs_cv/best_localizer_cv_metrics.json


,fold,heldout_log,num_observations,top1_exact_accuracy,top1_within1_accuracy,mean_abs_rank_error,median_abs_rank_error,mrr,pair_auc_roc,pair_average_precision
0,1,rep_bed-to-lavatory,115,0.652174,0.921739,0.426087,0.0,0.823188,0.993059,0.668025
1,2,rep_bed_tv,224,0.875000,0.941964,0.183036,0.0,0.936756,0.998823,0.891817
2,3,rep_dinning,265,0.701887,0.867925,0.430189,0.0,0.846541,0.997926,0.789409
3,4,rep_kitchen-long,253,0.841897,0.932806,0.229249,0.0,0.912846,0.998171,0.815265
4,5,rep_kitchen-short,150,0.926667,0.986667,0.086667,0.0,0.960000,0.998786,0.912265
5,6,rep_waiting_room,149,0.859060,0.959732,0.181208,0.0,0.927293,0.997595,0.844741
6,7,rep_wc,297,0.868687,0.932660,0.606061,0.0,0.934343,0.995560,0.868842


In [19]:

# ------------------------------------------------------------------
# ABSOLUTE RANK-ERROR DISTRIBUTION
# ------------------------------------------------------------------

rank_error_counts = (
    best_oof_observation_df["abs_rank_error"]
    .value_counts()
    .sort_index()
    .rename_axis("abs_rank_error")
    .reset_index(name="count")
)
display(rank_error_counts)

plt.figure(figsize=(7, 4))
plt.bar(rank_error_counts["abs_rank_error"], rank_error_counts["count"])
plt.xlabel("Absolute rank error")
plt.ylabel("Count")
plt.title("OOF absolute rank-error distribution")
plt.grid(True, axis="y")
plt.show()


,abs_rank_error,count
0,0,1195
1,1,156
2,2,100
3,3,1
4,123,1


In [20]:

# ------------------------------------------------------------------
# PER-LOG LOCALIZATION SUMMARY
# ------------------------------------------------------------------

per_log_summary_df = (
    best_oof_observation_df
    .groupby("source_log")
    .agg(
        num_observations=("raw_frame_id", "size"),
        top1_exact_accuracy=("is_exact_match", "mean"),
        top1_within1_accuracy=("is_within_1_rank", "mean"),
        mean_abs_rank_error=("abs_rank_error", "mean"),
        median_abs_rank_error=("abs_rank_error", "median"),
        mrr=("mrr", "mean"),
    )
    .reset_index()
)
display(per_log_summary_df)


,source_log,num_observations,top1_exact_accuracy,top1_within1_accuracy,mean_abs_rank_error,median_abs_rank_error,mrr
0,rep_bed-to-lavatory,115,0.652174,0.921739,0.426087,0.0,0.823188
1,rep_bed_tv,224,0.875000,0.941964,0.183036,0.0,0.936756
2,rep_dinning,265,0.701887,0.867925,0.430189,0.0,0.846541
3,rep_kitchen-long,253,0.841897,0.932806,0.229249,0.0,0.912846
4,rep_kitchen-short,150,0.926667,0.986667,0.086667,0.0,0.960000
5,rep_waiting_room,149,0.859060,0.959732,0.181208,0.0,0.927293
6,rep_wc,297,0.868687,0.932660,0.606061,0.0,0.934343


In [21]:

# ------------------------------------------------------------------
# ATTACH OOF PREDICTIONS TO OBSERVATION TABLE
# ------------------------------------------------------------------

oof_obs_df = best_oof_observation_df.copy()
oof_obs_df = oof_obs_df.sort_values(["source_log", "raw_frame_id"]).reset_index(drop=True)
display(oof_obs_df.head(20))


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
0,rep_bed-to-lavatory,0,3,1,3,1,0.270360,True,True,0,1.0,1,rep_bed-to-lavatory
1,rep_bed-to-lavatory,1,3,1,3,1,0.866090,True,True,0,1.0,1,rep_bed-to-lavatory
2,rep_bed-to-lavatory,2,3,1,4,2,0.803468,False,True,1,0.5,1,rep_bed-to-lavatory
3,rep_bed-to-lavatory,3,4,2,4,2,0.981689,True,True,0,1.0,1,rep_bed-to-lavatory
4,rep_bed-to-lavatory,4,6,3,6,3,0.151410,True,True,0,1.0,1,rep_bed-to-lavatory
5,rep_bed-to-lavatory,5,6,3,4,2,0.767225,False,True,1,0.5,1,rep_bed-to-lavatory
6,rep_bed-to-lavatory,6,8,4,8,4,0.480532,True,True,0,1.0,1,rep_bed-to-lavatory
7,rep_bed-to-lavatory,7,8,4,8,4,0.931339,True,True,0,1.0,1,rep_bed-to-lavatory
8,rep_bed-to-lavatory,8,10,5,10,5,0.620391,True,True,0,1.0,1,rep_bed-to-lavatory
9,rep_bed-to-lavatory,9,10,5,10,5,0.982246,True,True,0,1.0,1,rep_bed-to-lavatory


In [22]:

# ------------------------------------------------------------------
# VISUAL INSPECTION OF GOOD / BAD LOCALIZATIONS
# ------------------------------------------------------------------

def show_localization_row(row: pd.Series, loaded_logs: Dict[str, LoadedLog]):
    log_obj = loaded_logs[row["source_log"]]
    raw_frame_id = int(row["raw_frame_id"])
    pred_kf = int(row["pred_candidate_keyframe"])
    true_kf = int(row["target_keyframe"])

    observed_rgb = log_obj.rgb_frames[raw_frame_id]
    pred_rgb = log_obj.rgb_frames[pred_kf]
    true_rgb = log_obj.rgb_frames[true_kf]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(observed_rgb)
    axes[0].set_title(f"Observed RGB | frame {raw_frame_id}")
    axes[0].axis("off")

    axes[1].imshow(pred_rgb)
    axes[1].set_title(f"Predicted keyframe | {pred_kf}")
    axes[1].axis("off")

    axes[2].imshow(true_rgb)
    axes[2].set_title(f"True keyframe | {true_kf}")
    axes[2].axis("off")

    title = (
        f"log={row['source_log']} | exact={row['is_exact_match']} | "
        f"within1={row['is_within_1_rank']} | "
        f"target_rank={row['target_rank']} | pred_rank={row['pred_candidate_rank']} | "
        f"abs_rank_error={row['abs_rank_error']} | pred_score={row['pred_score']:.3f}"
    )
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

    display(pd.DataFrame([row]))


def show_top_localization_examples(
    df: pd.DataFrame,
    loaded_logs: Dict[str, LoadedLog],
    correct: bool = True,
    n: int = 4,
):
    subset = df[df["is_exact_match"] == correct].copy()
    if len(subset) == 0:
        print("No rows to show.")
        return

    if correct:
        subset = subset.sort_values(["pred_score", "mrr"], ascending=[False, False]).head(n)
    else:
        subset = subset.sort_values(["abs_rank_error", "pred_score"], ascending=[False, False]).head(n)

    print(f"Showing {'correct' if correct else 'wrong'} examples: {len(subset)}")
    for _, row in subset.iterrows():
        show_localization_row(row, loaded_logs)


In [23]:

# ------------------------------------------------------------------
# SOME GOOD LOCALIZATIONS
# ------------------------------------------------------------------

show_top_localization_examples(oof_obs_df, all_loaded_logs, correct=True, n=4)


Showing correct examples: 4


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
316,rep_bed_tv,201,202,93,202,93,1.0,True,True,0,1.0,2,rep_bed_tv


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
600,rep_dinning,261,262,126,262,126,1.0,True,True,0,1.0,3,rep_dinning


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
606,rep_kitchen-long,2,3,1,3,1,1.0,True,True,0,1.0,4,rep_kitchen-long


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
984,rep_kitchen-short,127,128,60,128,60,0.999989,True,True,0,1.0,5,rep_kitchen-short


In [24]:

# ------------------------------------------------------------------
# SOME BAD LOCALIZATIONS
# ------------------------------------------------------------------

show_top_localization_examples(oof_obs_df, all_loaded_logs, correct=False, n=6)


Showing wrong examples: 6


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
1425,rep_wc,269,272,123,0,0,0.0,False,False,123,1.0,7,rep_wc


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
741,rep_kitchen-long,137,138,69,135,66,0.765577,False,False,3,0.5,4,rep_kitchen-long


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
577,rep_dinning,238,240,112,236,110,0.90882,False,False,2,0.5,3,rep_dinning


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
561,rep_dinning,222,224,103,220,101,0.827909,False,False,2,0.5,3,rep_dinning


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
563,rep_dinning,224,226,104,222,102,0.825148,False,False,2,0.5,3,rep_dinning


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
522,rep_dinning,183,187,85,181,83,0.81729,False,False,2,0.5,3,rep_dinning


In [25]:

# ------------------------------------------------------------------
# MANUAL INSPECTION OF A SINGLE OOF OBSERVATION
# Edit row_index as needed.
# ------------------------------------------------------------------

row_index = 0
show_localization_row(oof_obs_df.iloc[row_index], all_loaded_logs)


,source_log,raw_frame_id,target_keyframe,target_rank,pred_candidate_keyframe,pred_candidate_rank,pred_score,is_exact_match,is_within_1_rank,abs_rank_error,mrr,fold,heldout_log
0,rep_bed-to-lavatory,0,3,1,3,1,0.27036,True,True,0,1.0,1,rep_bed-to-lavatory


## Runtime note

This notebook saves a **pair scorer**. In deployment, the runtime localizer should:

1. compute the same 9 pair features for the current observation against **every** visual-memory keyframe,
2. apply the saved scaler if present,
3. compute the positive-class score for every candidate,
4. select the candidate with the highest score.

That is the clean replacement for the current heuristic localizer.
